# Project 1 — Data Cleaning & Preparation

## Phase 3: Handling Missing Values

The purpose of this phase is to handle the missing values identified during the data-quality audit.

The audit found 309 missing values in the `CouponCode` column. Before making any changes, I will review those records and determine an appropriate way to handle the missing information while keeping useful records in the dataset.

In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("../data/raw/Dataset for Data Analytics.xlsx")

In [3]:
df.shape

(1200, 14)

In [4]:
df.isnull().sum()

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

In [5]:
clean_df = df.copy()

## Starting Point

The raw dataset contains 1,200 records and 14 columns. The Phase 2 audit identified 309 missing values in `CouponCode`. No other columns contain missing values.

The raw dataset will remain unchanged. All cleaning changes will be applied to a separate working copy.

## Investigating the Missing CouponCode Values

The `CouponCode` column contains 309 missing values. Before deciding how to handle them, I will inspect the affected records and compare them with records where a coupon code is present.

The goal is to understand whether the missing values represent missing information or a meaningful absence of a coupon.

In [6]:
missing_coupon_df = clean_df[
    clean_df["CouponCode"].isnull()
]

missing_coupon_df.head(10)

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
8,ORD200008,2025-04-02,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,NaN,Email,268.56
15,ORD200015,2023-07-17,C39416,Printer,1,473.96,942 Main St,Cash,Delivered,TRK54930938,3,NaN,Google,473.96
17,ORD200017,2024-03-02,C60672,Tablet,1,423.40,382 Main St,Debit Card,Shipped,TRK11972116,3,NaN,Referral,423.40
21,ORD200021,2024-11-17,C99023,Monitor,4,342.95,569 Main St,Cash,Delivered,TRK52646983,6,NaN,Google,1371.80
26,ORD200026,2024-03-06,C21944,Laptop,5,59.55,296 Main St,Online,Delivered,TRK80276812,6,NaN,Referral,297.75
28,ORD200028,2023-11-21,C59823,Phone,4,160.07,568 Main St,Online,Pending,TRK65895539,5,NaN,Email,640.28
31,ORD200031,2023-06-30,C26445,Chair,3,500.08,996 Main St,Debit Card,Returned,TRK31229981,6,NaN,Facebook,1500.24
32,ORD200032,2023-05-01,C12388,Tablet,5,536.72,830 Main St,Cash,Delivered,TRK99261395,10,NaN,Email,2683.60
36,ORD200036,2025-05-08,C59337,Laptop,2,534.43,465 Main St,Debit Card,Pending,TRK50961250,5,NaN,Facebook,1068.86
42,ORD200042,2024-05-06,C28712,Printer,2,151.66,980 Main St,Cash,Cancelled,TRK40772558,6,NaN,Instagram,303.32


In [7]:
missing_coupon_df.shape

(309, 14)

In [8]:
missing_coupon_df[
    [
        "OrderID",
        "Product",
        "Quantity",
        "UnitPrice",
        "PaymentMethod",
        "OrderStatus",
        "ItemsInCart",
        "CouponCode",
        "ReferralSource",
        "TotalPrice"
    ]
].head(20)

,OrderID,Product,Quantity,UnitPrice,PaymentMethod,OrderStatus,ItemsInCart,CouponCode,ReferralSource,TotalPrice
8,ORD200008,Phone,2,134.28,Gift Card,Cancelled,2,NaN,Email,268.56
15,ORD200015,Printer,1,473.96,Cash,Delivered,3,NaN,Google,473.96
17,ORD200017,Tablet,1,423.40,Debit Card,Shipped,3,NaN,Referral,423.40
21,ORD200021,Monitor,4,342.95,Cash,Delivered,6,NaN,Google,1371.80
26,ORD200026,Laptop,5,59.55,Online,Delivered,6,NaN,Referral,297.75
28,ORD200028,Phone,4,160.07,Online,Pending,5,NaN,Email,640.28
31,ORD200031,Chair,3,500.08,Debit Card,Returned,6,NaN,Facebook,1500.24
32,ORD200032,Tablet,5,536.72,Cash,Delivered,10,NaN,Email,2683.60
36,ORD200036,Laptop,2,534.43,Debit Card,Pending,5,NaN,Facebook,1068.86
42,ORD200042,Printer,2,151.66,Cash,Cancelled,6,NaN,Instagram,303.32


In [9]:
clean_df["CouponCode"].value_counts()

CouponCode
FREESHIP    313
WINTER15    292
SAVE10      286
Name: count, dtype: int64

In [10]:
clean_df["CouponCode"].value_counts(dropna=False)

CouponCode
FREESHIP    313
NaN         309
WINTER15    292
SAVE10      286
Name: count, dtype: int64

In [11]:
coupon_status = clean_df["CouponCode"].isnull().value_counts()

coupon_status

CouponCode
False    891
True     309
Name: count, dtype: int64

In [12]:
coupon_summary = pd.DataFrame({
    "status": ["Missing", "Present"],
    "count": [
        clean_df["CouponCode"].isnull().sum(),
        clean_df["CouponCode"].notnull().sum()
    ]
})

coupon_summary

,status,count
0,Missing,309
1,Present,891


In [13]:
clean_df.groupby(
    clean_df["CouponCode"].isnull()
)["TotalPrice"].mean()

CouponCode
False    1057.643715
True     1043.370259
Name: TotalPrice, dtype: float64

In [14]:
clean_df.groupby(
    clean_df["CouponCode"].isnull()
)["Quantity"].mean()

CouponCode
False    2.911336
True     3.045307
Name: Quantity, dtype: float64

In [15]:
clean_df.groupby(
    clean_df["CouponCode"].isnull()
)["ItemsInCart"].mean()

CouponCode
False    5.459035
True     5.559871
Name: ItemsInCart, dtype: float64

## Checking for Blank CouponCode Values

The null check identified 309 missing `CouponCode` values. I will also check for blank or whitespace-only values to make sure there are no other forms of missing information in the column.

In [16]:
blank_coupon_codes = (
    clean_df["CouponCode"]
    .dropna()
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

blank_coupon_codes

np.int64(0)

In [17]:
total_missing_coupon = (
    clean_df["CouponCode"].isnull().sum()
    + blank_coupon_codes
)

total_missing_coupon

np.int64(309)

## Treatment Decision

The investigation found 309 missing `CouponCode` values and no additional blank or whitespace-only values.

The missing values appear in otherwise complete order records, and the dataset does not contain a separate field indicating whether a coupon was used. For this project, I will treat a missing `CouponCode` as an order where no coupon was recorded and replace the missing value with `NO_COUPON`.

This decision keeps all order records while making the meaning of the missing value explicit and easier to use in later analysis.

In [18]:
rows_before = len(clean_df)
missing_coupon_before = clean_df["CouponCode"].isnull().sum()

rows_before, missing_coupon_before

(1200, np.int64(309))

## Handling the Missing CouponCode Values

The missing `CouponCode` values will now be replaced with `NO_COUPON`. The change will be applied only to the working copy of the dataset.

In [19]:
clean_df["CouponCode"] = clean_df["CouponCode"].fillna("NO_COUPON")

In [20]:
clean_df["CouponCode"].isnull().sum()

np.int64(0)

In [21]:
clean_df["CouponCode"].value_counts(dropna=False)

CouponCode
FREESHIP     313
NO_COUPON    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64

In [22]:
rows_after = len(clean_df)

rows_before, rows_after

(1200, 1200)

In [23]:
values_changed = (
    df["CouponCode"].isnull()
    & clean_df["CouponCode"].eq("NO_COUPON")
).sum()

values_changed

np.int64(309)

In [24]:
df

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ORD201195,2024-06-20,C21126,Desk,1,107.04,392 Main St,Credit Card,Cancelled,TRK38009181,6,FREESHIP,Google,107.04
1196,ORD201196,2024-03-04,C20095,Monitor,2,662.53,778 Main St,Online,Cancelled,TRK69207593,5,NaN,Facebook,1325.06
1197,ORD201197,2023-07-13,C79674,Tablet,2,436.84,275 Main St,Online,Delivered,TRK88039356,2,FREESHIP,Instagram,873.68
1198,ORD201198,2024-08-22,C64753,Chair,4,262.52,509 Main St,Debit Card,Cancelled,TRK71683331,4,WINTER15,Instagram,1050.08


## Raw Dataset Check

The original `df` dataframe should remain unchanged. All cleaning was applied to `clean_df`, so the raw missing-value count should still be 309.

In [25]:
df["CouponCode"].isnull().sum()

np.int64(309)

In [26]:
clean_df["CouponCode"].isnull().sum()

np.int64(0)

## Final Missing Value Check

After handling the missing `CouponCode` values, I will check the complete working dataset again to confirm that no null values remain.

In [27]:
final_missing = clean_df.isnull().sum()

final_missing

OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64

In [28]:
final_missing.sum()

np.int64(0)

In [29]:
clean_df.shape

(1200, 14)

In [30]:
clean_df.columns.tolist()

['OrderID',
 'Date',
 'CustomerID',
 'Product',
 'Quantity',
 'UnitPrice',
 'ShippingAddress',
 'PaymentMethod',
 'OrderStatus',
 'TrackingNumber',
 'ItemsInCart',
 'CouponCode',
 'ReferralSource',
 'TotalPrice']

## Exporting the Cleaned Dataset

The cleaned working dataframe will now be saved as a separate Excel file in the `processed` folder. The original raw dataset will remain unchanged.

In [31]:
output_path = "../data/processed/cleaned_dataset.xlsx"

clean_df.to_excel(output_path, index=False)

In [32]:
saved_df = pd.read_excel(output_path)

saved_df.shape

(1200, 14)

In [33]:
saved_df.isnull().sum().sum()

np.int64(0)

In [34]:
saved_df["CouponCode"].value_counts(dropna=False)

CouponCode
FREESHIP     313
NO_COUPON    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64

## Phase 3 Summary

The missing-value audit identified 309 missing values in the `CouponCode` column.

After reviewing the affected records, the missing values were treated as `NO_COUPON` because the dataset does not provide a separate field indicating coupon usage and the affected records otherwise contain complete order information.

The replacement was applied to a copy of the raw dataset rather than the original source data. All 1,200 records were retained, no columns were removed, and the final cleaned dataset contains zero missing values.

The cleaned dataset was saved to `data/processed/cleaned_dataset.xlsx`.